## Summary 

This notebook takes the milk recordings and fits the MilkBot lactation curve model using the lactation curve package. The fitted characteristics are saved and used downstream.

# Bi-modal milk flow project, calculate lactation curve characteristics 

Notebook contains the following
- part 2 of data preprocessing farm data
- Calculation of lactation curve characteristics 
- Calculation of proportion of bi-modal-milk flow (DME)

Calculated characteristics are saved, as it takes a lot of memory and time to calculate them for each lactation.

Import packages

lactation curve package: van Leerdam, M. B., de Kok, D., Osei-Tete, J. A., & Hostens, M. (2026). Bovi-analytics/bovi lactationcurve: v.1.3.1 (Version 1.3.1) [Computer software]. DOI: 10.5281/zenodo.23898172

In [0]:
%pip install lactationcurve

from lactationcurve.characteristics import calculate_characteristic
from lactationcurve.fitting import fit_lactation_curve

In [0]:
#try out if the import of the function went alright 

dim = [10,20,30]
y = [20,35,40]

print(calculate_characteristic(dim, y, model = 'MilkBot', characteristic = 'time_to_peak'))
print(calculate_characteristic(
        dim,
        y,
        model = 'wood',
        characteristic = 'persistency',
        fitting='frequentist',
        persistency_method = 'derived'))

In [0]:
#other packages
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
from pyspark.sql.window import Window
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
# from statsmodels.formula.api import ols
from scipy.stats import norm 
# import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

Connect data

# Data preparation

In [0]:
#get rid of yesterday in the column name as it is no longer relevant and causes confusion
bi_modal_df = bi_modal_df.toDF(*[
    c.replace('yesterday', '').replace('Yesterday', '') 
    for c in bi_modal_df.columns
])

#drop two rows where DIM = 0 
bi_modal_df = bi_modal_df.filter(bi_modal_df['DaysInMilk'].isNotNull())

# Calculate CalvingDate and CalvingSeason for each lactation
bi_modal_df = (
    bi_modal_df
    .withColumn(
        "LastMilkDate",
        F.to_timestamp("LastMilkTime", "M/d/yyyy h:mm a")
    )
    .withColumn(
        "CalvingDate",
        F.date_sub(F.col("LastMilkDate"), F.col("DaysInMilk"))
    )
    .withColumn(
        "CalvingSeason",
        F.when(F.month("CalvingDate").isin(12, 1, 2), "Winter")
         .when(F.month("CalvingDate").isin(3, 4, 5), "Spring")
         .when(F.month("CalvingDate").isin(6, 7, 8), "Summer")
         .otherwise("Autumn")
    )
    .withColumn(
        'CalvingYear',
        F.year("CalvingDate")
    )
)

#somehow there has been an introduction of dulpicate rows, erase them again 
bi_modal_df = bi_modal_df.dropDuplicates()



In [0]:
#create criteria to filter lactations so we get less weird lactation characteristics 

# They must have minimal 10 measurements 
# -----------------------------
# 1. Count measurements per lactation
# -----------------------------
counts = (
    bi_modal_df.groupBy("LactationId")
      .agg(F.count("*").alias("n_rows"))
      .filter(F.col("n_rows") >= 10)
)

# -----------------------------
# 2. Check presence of early and late DIM measurements 
# There must be a milkrecording in the first 50 days and in the last 50 days of the lactation 
# -----------------------------
dim_flags = (
    bi_modal_df.groupBy("LactationId")
      .agg(
          F.max(F.col("DaysInMilk") <= 50).cast("int").alias("has_early"),
          F.max((F.col("DaysInMilk") >= 255) & (F.col("DaysInMilk") <= 305))
            .cast("int").alias("has_late")
      )
      .filter("has_early = 1 AND has_late = 1")
)

# -----------------------------
# 3. Keep only lactations that satisfy ALL criteria
# -----------------------------
valid_lactations = (
    counts.join(dim_flags, on="LactationId", how="inner")
          .select("LactationId")
)

filtered_df = (
    bi_modal_df.join(valid_lactations, on="LactationId", how="inner")
)


In [0]:
#Descriptive stats df
number_of_recordings = bi_modal_df.count()
number_of_animals = bi_modal_df.select('AnimalNumber').distinct().count()
number_of_parities = bi_modal_df.select('LactationNumber').distinct().count()
number_of_lactations = bi_modal_df.select('LactationId').distinct().count()
print(f'Number of recordings: {number_of_recordings}')
print(f'Number of animals: {number_of_animals}')
print(f'Number of parities: {number_of_parities}')
print(f'Number of lactations: {number_of_lactations}')
print(f'Average amount of lactations per animal: {number_of_lactations/number_of_animals}')

In [0]:
#Descriptive stats df
number_of_recordings = filtered_df.count()*3
number_of_animals = filtered_df.select('AnimalNumber').distinct().count()
number_of_parities = filtered_df.select('LactationNumber').distinct().count()
number_of_lactations = filtered_df.select('LactationId').distinct().count()
print(f'Number of recordings: {number_of_recordings}')
print(f'Number of animals: {number_of_animals}')
print(f'Number of parities: {number_of_parities}')
print(f'Number of lactations: {number_of_lactations}')
print(f'Average amount of lactations per animal: {number_of_lactations/number_of_animals}')

## Seperate Bi-midal from non-bi-modal 


In [0]:
#create a variable for proportion of DME per lactation 
#create a window per lactation 
w = Window.partitionBy("LactationId")

#Number of milkings with observations (assuming no missing milk measurements)
filtered_df = filtered_df.withColumn(
    "Valid_sessions",
    F.lit(3)
)

# with missing milk measurements
# subset = subset.withColumn(
#     "Valid_sessions",
#     F.col("Flow30_60Session1Yesterday").isNotNull().cast("int") +
#     F.col("Flow30_60Session2Yesterday").isNotNull().cast("int") +
#     F.col("Flow30_60Session3Yesterday").isNotNull().cast("int")
# )

#Number of milkings with bimodal milk flow curve 
filtered_df = filtered_df.withColumn(
    "BIM_sessions",
    F.col("BIM_1").cast("int") +
    F.col("BIM_2").cast("int") +
    F.col("BIM_3").cast("int")
)

filtered_df = filtered_df.withColumn(
    "BIM_prop",
    F.sum("BIM_sessions").over(w) / F.sum("Valid_sessions").over(w)
)

#rename column BIM to DME 
filtered_df = filtered_df.withColumnRenamed('BIM_prop', 'DME_prop')
filtered_df = filtered_df.withColumnRenamed('BIM_prop_logit', 'DME_prop_logit')

#split based on a threshold 
bins = [0, 0.20, 0.40, 1.0]
labels = ["low", "middle", "high"]

filtered_df = filtered_df.withColumn(
    "DME_prop_cat",
    F.when(F.col("DME_prop") <= bins[1], labels[0])
     .when((F.col("DME_prop") > bins[1]) & (F.col("DME_prop") <= bins[2]), labels[1])
     .when((F.col("DME_prop") > bins[2]) & (F.col("DME_prop") <= bins[3]), labels[2])
)

### Calculate proportion of DME only at the beginning of lactation

In [0]:
cut_off = 150

# Window for second half of lactation only (DaysInMilk <= cut_off)
w_second_half = Window.partitionBy("LactationId")

# --- second half of lactation DME proportion (DaysInMilk <= cut_off) ---
filtered_df = filtered_df.withColumn(
    "DME_sessions_second_half",
    F.when(F.col("DaysInMilk") >= cut_off, F.col("BIM_sessions")).otherwise(0)
)
filtered_df = filtered_df.withColumn(
    "Valid_sessions_second_half",
    F.when(F.col("DaysInMilk") >= cut_off, F.col("Valid_sessions")).otherwise(0)
)
filtered_df = filtered_df.withColumn(
    "DME_prop_second_half",
    F.sum("DME_sessions_second_half").over(w_second_half) / F.sum("Valid_sessions_second_half").over(w_second_half)
)

# Drop helper columns
filtered_df = filtered_df.drop("DME_sessions_second_half", "Valid_sessions_second_half")

#split based on a threshold 
bins = [0, 0.20, 0.40, 1.0]
labels = ["low", "middle", "high"]

# second half of lactation DME category
filtered_df = filtered_df.withColumn(
    "DME_prop_second_half_cat",
    F.when(F.col("DME_prop_second_half") <= bins[1], labels[0])
     .when((F.col("DME_prop_second_half") > bins[1]) & (F.col("DME_prop_second_half") <= bins[2]), labels[1])
     .when((F.col("DME_prop_second_half") > bins[2]) & (F.col("DME_prop_second_half") <= bins[3]), labels[2])
)



In [0]:
#alternative create a histplot as the visualisation of databricks somethimes dissapoints 

data = filtered_df.toPandas()

# data['BIM_prop'].plot.hist(bins=100, title="BIM proportion per lactation")
plt.show()


In [0]:
# Add the square root column to the pandas DataFrame
data['SqrtBMIProp'] = data['DME_prop'].apply(lambda x: np.sqrt(x))

plt.hist(
    data["SqrtBMIProp"].dropna(),
    bins=100
)
plt.title("sqrt(DME proportion) per lactation")
plt.show()

# Only apply log to positive values to avoid -inf and NaN
data['logBMIProp'] = data['DME_prop'].apply(
    lambda x: np.log(x) if x > 0 else 0
)

plt.hist(
    data["logBMIProp"].dropna(),
    bins=100
)
plt.title("log(DME proportion) per lactation")
plt.show()

eps = 1e-6

data["DME_prop_logit"] = np.log(
    (data["DME_prop"] + eps) / (1 - data["DME_prop"] + eps)
)
plt.hist(
    data["DME_prop_logit"],
    bins=100
)
plt.title("logit(DME proportion) per lactation")
plt.show()

# Arcsine-sqrt transform
data["DME_prop_arcsine"] = np.arcsin(np.sqrt(data["DME_prop"]))

plt.hist(
    data["DME_prop_arcsine"],
    bins=100
)
plt.title("arcsine(DME proportion) per lactation")
plt.show()

In [0]:
#calculate mean and 95% c.i. of DME prop

mean = data['DME_prop'].mean()
sem = data['DME_prop'].std(ddof=1) / np.sqrt(len(data['DME_prop']))

ci_low, ci_high = stats.t.interval(
    confidence=0.95,
    df=len(data['DME_prop'])-1,
    loc=mean,
    scale=sem
)

print('The mean DME proportion is: ', mean)
print('The standard error of the mean is: ', sem)
print('The 95% confidence interval is: ', ci_low, ci_high)

## Calculate lactationcurve characteristic 

In [0]:
#Use the lactationcurve characteristic part of the lc package to calculate LC characteristics 
#first convert df to pandas
# use subset for now
bim_pd = data

time_to_peak_list = []
lactation_id = []
peak_yield_list = []
cummulative_yield_list = []
persistency_list = []


#create a loop to calculate all the characteristics for each LactationId
for lac_id in bim_pd['LactationId'].unique():
    dim = bim_pd[bim_pd['LactationId'] == lac_id]['DaysInMilk'].values
    milk_yield = bim_pd[bim_pd['LactationId'] == lac_id]['DailyMilkYield'].values
    try:
        time_to_peak = calculate_characteristic(
            dim,
            milk_yield,
            model='milkbot',
            characteristic='time_to_peak',
            fitting='frequentist'
        )
    except Exception:
        time_to_peak = None
    peak_yield = calculate_characteristic(
        dim,
        milk_yield,
        model = 'milkbot',
        characteristic = 'peak_yield',
        fitting='frequentist'
        )
    cummulative_yield = calculate_characteristic(
        dim,
        milk_yield,
        model = 'milkbot',
        characteristic = 'cumulative_milk_yield',
        fitting='frequentist'
        )
    persistency = calculate_characteristic(
        dim,
        milk_yield,
        model = 'milkbot',
        characteristic = 'persistency',
        fitting='frequentist',
        persistency_method = 'literature'
        )
    time_to_peak_list.append(time_to_peak)
    lactation_id.append(lac_id)
    peak_yield_list.append(peak_yield)
    cummulative_yield_list.append(cummulative_yield)
    persistency_list.append(persistency)

# Create dataframe 

characteristics = pd.DataFrame(
    list(zip(lactation_id, time_to_peak_list, peak_yield_list, cummulative_yield_list, persistency_list)),
    columns=['LactationId', 'TimeToPeak', 'PeakYield', 'CummulativeYield', 'Persistency']
)

#join the dataframes together 
df_final = bim_pd.merge(characteristics, on='LactationId', how='right')

#group by lactation id 
df_final = df_final.groupby('LactationId', as_index=False).first()


In [0]:
#drop all rows with a null for time to peak and a negative cummulative milk yield 
df_cleaned = df_final[
    df_final["TimeToPeak"].notna() & 
    (df_final["TimeToPeak"] < 305) & #You cannot peak after you finished your lactation 
    # (df_final["Persistency"] < 10000) & #gut feeling 
    (df_final['PeakYield'] < 127) #world record milk kg per day 
]


In [0]:
#count of lactation lost due to final cleaning step 
print(df_final.shape[0] - df_cleaned.shape[0])

#print the LactationId of lost lactations
df_final.select('LactationId').subtract(df_cleaned.select('LactationId')).show(100)



In [0]:
#visualize model variables 
plt.hist(
    df_final['TimeToPeak'],
    bins=100
)
plt.title('Time to peak distribution')
plt.xlabel('Time to peak')
plt.ylabel('Frequency')
plt.show()

plt.hist(df_final['PeakYield'], bins=100)
plt.title('Peak distribution')
plt.xlabel('Peak')
plt.ylabel('Frequency')
plt.show()

plt.hist(df_final['CummulativeYield'], bins=100)
plt.title('Cummulative yield distribution')
plt.xlabel('Cummulative yield')
plt.ylabel('Frequency')
plt.show()

plt.hist(df_final['Persistency'], bins=100)
plt.title('Persistency distribution')
plt.xlabel('Persistency')
plt.ylabel('Frequency')
plt.show()

plt.hist(df_final['DailyMilkYield'], bins = 100)
plt.title('Daily milk yield distribution')
plt.xlabel('Daily milk yield')
plt.ylabel('Frequency')
plt.show

## Add the DME proportion for the first half of the lactation

In [0]:
#add from the filtered_df the column DME_prop_second_half_cat and DME_prop_second_half

dme_second_half = filtered_df.select('LactationId', 'DME_prop_second_half_cat', 'DME_prop_second_half').dropDuplicates(['LactationId'])

characteristics_df = characteristics_df.join(dme_second_half, on='LactationId', how='left')

def logit(col_name):
    eps = F.lit(1e-6)
    c = F.col(col_name)
    return F.log((c + eps) / (F.lit(1) - c + eps))

#add logit transformed version of DME_prop_second_half
characteristics_df = characteristics_df.withColumn('logit_DME_prop_second_half', logit('DME_prop_second_half'))

display(characteristics_df)

#count number of rows
print(characteristics_df.count())
                                             

## create lactationcurve visualisations of bimodality 

In [0]:

#split based on a threshold 
bins = [0, 0.20, 0.40, 1.0]
labels = ["low", "middle", "high"]

data["DME_prop_cat"] = pd.cut(data["DME_prop"], bins=bins, labels=labels, include_lowest=True)

#convert lactationnumber to 4 categories, 1,2,3 where everything above 3 becomes 3. 
data["LactationCat"] = data["LactationNumber"].clip(upper=3)

#groupby lactationId to get one observation per lactation
observations = data.groupby('LactationId').first()

In [0]:
#alternative create a categorial variable out of BMI prop 

# Create a plot to visualize the number of lactations per category
counts = observations['DME_prop_cat'].value_counts().reindex(["low", "middle", "high"])
plt.bar(counts.index, counts.values, color=[plt.cm.Reds(0.4), plt.cm.Reds(0.7), plt.cm.Reds(1.0)])

# Print the column name at the middle of each bar
for i, (cat, count) in enumerate(counts.items()):
    plt.text(i, count / 2, str(count), ha='center', va='center', fontsize=10, color='black')

plt.title('DME proportion distribution')
plt.xlabel('DME proportion')
plt.ylabel('Frequency')
plt.show()



### Number of obeservations per Parity and DME category

In [0]:

#count number of LactationCat 
counts = observations['LactationCat'].value_counts().reindex([1, 2, 3])
plt.bar(counts.index, counts.values, color=[plt.cm.Greens(0.4), plt.cm.Greens(0.7), plt.cm.Greens(1.0)])

# Print the column name at the middle of each bar
for i, (cat, count) in enumerate(counts.items()):
    plt.text(i, count / 2, str(count), ha='center', va='center', fontsize=10, color='black')

plt.title('Lactation number distribution')
plt.xlabel('Lactation number')
plt.ylabel('Frequency')
plt.show()

In [0]:
#make overview of number of lactations per parity per categroy

for parity in observations['LactationCat'].unique():
    counts = observations[observations['LactationCat'] == parity]['DME_prop_cat'].value_counts()
    print(f"Parity {parity}:")
    print(counts)
    print("\n")


### fit a lactationcurve per DME category 

In [0]:
filtered_df = filtered_df.filter(filtered_df['DaysInMilk'] < 305)
display(filtered_df)

In [0]:

parity_1 = filtered_df.filter(filtered_df['LactationNumber'] == 1)

display(parity_1)

# To change colors in Databricks visualizations, click the "Edit Visualization" button after running display(),
# then manually set the color for each DME_prop_cat value in the visualization editor.

In [0]:
#make a matplotlib version of an average milk yield per days in milk for each parity
fig, ax = plt.subplots(figsize=(14/2.54, 10/2.54))  # 14 cm wide
for parity in sorted(data['LactationCat'].unique()): 
    subset = data[data['LactationCat'] == parity]
    #groupby days in milk 
    subset = subset.groupby('DaysInMilk').mean('DailyMilkYield')
    label = f'{parity}+' if parity == 3 else str(parity)
    ax.plot(subset['DailyMilkYield'], label=label)
ax.legend(fontsize=14, title='Parity')
# plt.title('Average milk yield per days in milk for each parity')
ax.set_xlabel('Days in milk', fontsize=14)
ax.set_ylabel('Average milk yield', fontsize=14)
plt.tight_layout()
plt.show()
     


In [0]:
#make a matplotlib version of an average milk yield per days in milk for each DME prop category
fig, ax = plt.subplots(figsize=(19/2.54, 10/2.54))  # 19 cm wide
for dme_prop in sorted(data['DME_prop_cat'].unique()): 
    subset = data[data['DME_prop_cat'] == dme_prop]
    #groupby days in milk 
    subset = subset.groupby('DaysInMilk').mean('DailyMilkYield')
    label = f'{dme_prop}+' if dme_prop == 3 else str(dme_prop)
    ax.plot(subset['DailyMilkYield'], label=label)
handles, labels = ax.get_legend_handles_labels()
order = ['low', 'middle', 'high']
ordered_handles = [handles[labels.index(o)] for o in order]
ordered_labels = order
# ax.legend(ordered_handles, ordered_labels, fontsize=13, title='DME proportion', title_fontsize=13)
# plt.title('Average milk yield per days in milk for each dme_prop')
ax.set_xlabel('Days in milk', fontsize=14.5)
ax.set_ylabel('Average milk yield (kg/day)', fontsize=14.5)
ax.tick_params(axis='both', labelsize=14)
plt.tight_layout()
plt.savefig("average_milk_yield_per_dim_by_DME_category.png", dpi=400, bbox_inches="tight")
plt.show()

import base64
for fname in ["average_milk_yield_per_dim_by_DME_category.png"]:
    with open(fname, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    displayHTML(f'<a href="data:image/png;base64,{b64}" download="{fname}">{fname} (400 DPI)</a><br>')

In [0]:
#parity 1, 2, 3
parity_1 = data[data['LactationCat'] == 1]
parity_2 = data[data['LactationCat'] == 2]
parity_3 = data[data['LactationCat'] == 3]

for parity_num, parity_df in [(1, parity_1), (2, parity_2), (3, parity_3)]:
    fig, ax = plt.subplots(figsize=(19/2.54, 10/2.54))  # 19 cm wide
    for dme_prop in sorted(parity_df['DME_prop_cat'].unique()): 
        subset = parity_df[parity_df['DME_prop_cat'] == dme_prop]
        #groupby days in milk 
        subset = subset.groupby('DaysInMilk').mean('DailyMilkYield')
        label = str(dme_prop)
        ax.plot(subset['DailyMilkYield'], label=label)
    handles, labels = ax.get_legend_handles_labels()
    order = ['low', 'middle', 'high']
    ordered_handles = [handles[labels.index(o)] for o in order]
    ordered_labels = order
    # ax.legend(ordered_handles, ordered_labels, fontsize=12, title='DME proportion', title_fontsize=12)
    parity_label = f'{parity_num}+' if parity_num == 3 else str(parity_num)
    # ax.set_title(f'Parity {parity_label}', fontsize=14.5)
    ax.set_xlabel('Days in milk', fontsize=14.5)
    ax.set_ylabel('Average milk yield (kg/day)', fontsize=14.5)
    ax.tick_params(axis='both', labelsize=14)
    plt.tight_layout()
    fname = f"average_milk_yield_per_dim_by_DME_category_parity_{parity_num}.png"
    plt.savefig(fname, dpi=400, bbox_inches="tight")
    plt.show()

import base64
for parity_num in [1, 2, 3]:
    fname = f"average_milk_yield_per_dim_by_DME_category_parity_{parity_num}.png"
    with open(fname, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    displayHTML(f'<a href="data:image/png;base64,{b64}" download="{fname}">{fname} (400 DPI)</a><br>')


In [0]:
parity_2 = filtered_df.filter(filtered_df['LactationCat'] == 2)
display(parity_2)

In [0]:
parity_3 = filtered_df.filter(filtered_df['LactationCat'] == 3)
display(parity_3)

In [0]:
#convert to pandas for lc modelling
lc_df_panda = filtered_df.toPandas()

fit_lc = {}

for cat in lc_df_panda['DME_prop_cat'].unique():
    subset = lc_df_panda[lc_df_panda['DME_prop_cat'] == cat]
    dim = subset['DaysInMilk']
    my = subset["DailyMilkYield"]
    fit = fit_lactation_curve(dim, my, model = 'milkbot', fitting = 'frequentist')
    
    fit_lc[cat] = fit

#plot 
cat_order = ["high", "middle", "low"]
legend_labels = {
    "high": "high (>40%)",
    "middle": "middle (20-40%)",
    "low": "low (<20%)"
}
red_palette = [plt.cm.Reds(1.0), plt.cm.Reds(0.7), plt.cm.Reds(0.4)]
for i, cat in enumerate(cat_order):
    if cat in fit_lc:
        plt.plot(fit_lc[cat], label=legend_labels[cat], color=red_palette[i])
plt.legend()
plt.title('Lactation curve per DME proportion category')
plt.xlabel('Days in milk')
plt.ylabel('Milk yield (kg/day)')
plt.show()
 
 

### plot LC per parity per DME 

In [0]:
red_palette = [plt.cm.Reds(1.0), plt.cm.Reds(0.7), plt.cm.Reds(0.4)]
cat_order = ["high", "middle", "low"]

for lac_cat in data['LactationCat'].unique():
    subset_lac = data[data['LactationCat'] == lac_cat]
    fit_lc = {}

    for cat in cat_order:
        if cat in subset_lac['DME_prop_cat'].unique():
            subset = subset_lac[subset_lac['DME_prop_cat'] == cat]
            dim = subset['DaysInMilk']
            my = subset["DailyMilkYield"]
            fit = fit_lactation_curve(dim, my, model='milkbot', fitting='frequentist')
            fit_lc[cat] = fit

    for i, cat in enumerate(cat_order):
        if cat in fit_lc:
            plt.plot(fit_lc[cat], label=cat, color=red_palette[i])
    plt.legend()
    if lac_cat == 3:
        plt.title(f'Lactation curve per DME proportion category for parity {lac_cat}+')
    else:
        plt.title(f'Lactation curve per DME proportion category for parity {lac_cat}')
    plt.xlabel('Days in milk')
    plt.ylabel('Milk yield (kg/day)')
    plt.show()

## changes in DME over time 

In [0]:
# Histogram of time to peak per lactation, split by parity
bi_modal_pd = bi_modal_df.toPandas()

# Create LactationCat: 1, 2, 3+
bi_modal_pd['LactationCat'] = bi_modal_pd['LactationNumber'].apply(lambda x: 3 if x > 3 else x)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for i, lac_cat in enumerate([1, 2, 3]):
    subset = bi_modal_pd[bi_modal_pd['LactationCat'] == lac_cat]
    label = f'Parity {lac_cat}+' if lac_cat == 3 else f'Parity {lac_cat}'
    axes[i].hist(subset['TimeToPeak'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_xlabel('Time to peak (days)', fontsize=14)
    axes[i].set_title(label, fontsize=14)
    axes[i].tick_params(axis='both', labelsize=12)

axes[0].set_ylabel('Number of lactations', fontsize=14)
plt.tight_layout()
plt.show()

In [0]:
data_filtered_lc = data[data['LactationId'].isin(lactation_ids)]
number_of_recordings = len(data_filtered_lc) * 3
number_of_animals = data_filtered_lc['AnimalNumber'].nunique()
number_of_parities = data_filtered_lc['LactationNumber'].nunique()
number_of_lactations = data_filtered_lc['LactationId'].nunique()
print(f'Number of recordings: {number_of_recordings}')
print(f'Number of animals: {number_of_animals}')
print(f'Number of parities: {number_of_parities}')
print(f'Number of lactations: {number_of_lactations}')
print(f'Average amount of lactations per animal: {number_of_lactations/number_of_animals}')

In [0]:
# Histogram of valid milkings per LactationId
valid_per_lac = data_filtered_lc.groupby('LactationId')['Valid_sessions'].sum()

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(valid_per_lac.values, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Number of valid milkings', fontsize=14)
ax.set_ylabel('Number of lactations', fontsize=14)
ax.set_title('Distribution of valid milkings per lactation', fontsize=14)
ax.tick_params(axis='both', labelsize=12)
plt.tight_layout()
plt.show()

In [0]:
data_305 = data_filter_lc[data_filter_lc['DaysInMilk'] <= 305]
number_of_recordings = len(data_305) * 3
number_of_animals = data_305['AnimalNumber'].nunique()
number_of_parities = data_305['LactationNumber'].nunique()
number_of_lactations = data_305['LactationId'].nunique()
print(f'Number of recordings: {number_of_recordings}')
print(f'Number of animals: {number_of_animals}')
print(f'Number of parities: {number_of_parities}')
print(f'Number of lactations: {number_of_lactations}')
print(f'Average amount of lactations per animal: {number_of_lactations/number_of_animals}')

In [0]:
#calculate sum of all valid sessions
data_filtered_lc['Valid_sessions'].sum()


In [0]:
#Descriptive stats on pandas 'data_305' df
number_of_recordings = len(data_305) * 3
number_of_animals = data_305['AnimalNumber'].nunique()
number_of_parities = data_305['LactationNumber'].nunique()
number_of_lactations = data_305['LactationId'].nunique()
print(f'Number of recordings: {number_of_recordings}')
print(f'Number of animals: {number_of_animals}')
print(f'Number of parities: {number_of_parities}')
print(f'Number of lactations: {number_of_lactations}')
print(f'Average amount of lactations per animal: {number_of_lactations/number_of_animals}')

In [0]:
#lost of measurements because of taking the values out below 305 
diff = len(data)-len(data_305) 
prop = diff/len(data)
print(f'Lost of measurements: {diff} ({prop*100:.2f}%)')


In [0]:
#Print a histogram of DaysInMilk 
data['DaysInMilk'].hist(bins=30)
plt.title('Histogram of DaysInMilk')
plt.xlabel('Days in milk')
plt.ylabel('Frequency')
plt.show()

In [0]:
#cut-off at dim 305 
data_305 = data_filter_lc[data_filter_lc['DaysInMilk'] < 305]

grouped = data_305.groupby('DaysInMilk')[['BIM_sessions', 'Valid_sessions']].sum()
prop_DME_per_day = grouped['BIM_sessions'] / grouped['Valid_sessions']
total_measurements = grouped['Valid_sessions']

fig, ax1 = plt.subplots(figsize=(12, 6))

# Bar plot for proportion of DME (left axis)
ax1.bar(prop_DME_per_day.index, prop_DME_per_day.values, width=1, color='steelblue', alpha=0.7, label='Proportion of DME')
ax1.set_xlabel('Days in milk')
ax1.set_ylabel('Proportion of DME', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

# Line plot for amount of measurements (right axis)
ax2 = ax1.twinx()
ax2.plot(total_measurements.index, total_measurements.values, color='orange', linewidth=1.5, label='Number of measurements')
ax2.set_ylabel('Number of measurements', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Proportion of milkings with DME per day versus the number of milkings per day')
fig.tight_layout()
plt.show()

#plot is cut-off at dim 305 

In [0]:
#all data
grouped = data_filter_lc.groupby('DaysInMilk')[['BIM_sessions', 'Valid_sessions']].sum()
prop_DME_per_day = grouped['BIM_sessions'] / grouped['Valid_sessions']

prop_DME_per_day.plot()
plt.title('Proportion of milkings with DME per day for all data')
plt.xlabel('Days in milk')
plt.ylabel('Proportion of DME')
plt.show()

In [0]:
#cut-off at dim 305 
data_305 = data[data['DaysInMilk'] < 305]

#alternative if you drop day 1
data_2_to_305 = data_305[data_305['DaysInMilk'] > 3]

grouped = data_2_to_305.groupby('DaysInMilk')[['BIM_sessions', 'Valid_sessions']].sum()
prop_DME_per_day = grouped['BIM_sessions'] / grouped['Valid_sessions']

prop_DME_per_day.plot()
plt.title('Proportion of milkings with DME per day day 1 dropped')
plt.xlabel('Days in milk')
plt.ylabel('Proportion of DME')
plt.show()

In [0]:
#make a graph of the amount of measurements per DaysInMilk 
grouped.plot()
plt.title('Amount of measurements per day')
plt.xlabel('Days in milk')
plt.ylabel('Amount of measurements')
plt.show()


In [0]:
import base64
colors = {1: '#0072B2', 2: '#E69F00', 3: '#009E73'}  # Colorblind-friendly palette (blue, orange, green)

#set font
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["DejaVu Serif"]

plt.figure(figsize=(10, 6))
for lac_cat in sorted(data_305['LactationCat'].unique()):
    subset_lac = data_305[data_305['LactationCat'] == lac_cat]
    grouped = subset_lac.groupby('DaysInMilk')[['BIM_sessions', 'Valid_sessions']].sum()
    prop_DME_per_day = grouped['BIM_sessions'] / grouped['Valid_sessions']
    if lac_cat == 3: 
        plt.plot(prop_DME_per_day.index, prop_DME_per_day.values, label=f'Parity {lac_cat}+', color=colors.get(lac_cat, None))
    else: 
        plt.plot(prop_DME_per_day.index, prop_DME_per_day.values, label=f'Parity {lac_cat}', color=colors.get(lac_cat, None))

# plt.title('Proportion of milkings with DME per day by parity')

plt.xlabel('Days in milk',fontsize=14)
plt.ylabel('Proportion of DME',fontsize=14)
plt.legend(fontsize=14)
plt.savefig("Proportion_of_milkings_with_DME_per_day_by_parity", dpi=400, bbox_inches="tight")


# Create download links
for fname in ["Proportion_of_milkings_with_DME_per_day_by_parity.png"]:
    with open(fname, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    displayHTML(f'<a href="data:image/png;base64,{b64}" download="{fname}">{fname} (400 DPI)</a><br>')